# 06 — Lancer la démonstration Colab

**Statut : exécuté et vérifié le 10 novembre 2026 sur une L4 Colab (médiane HTTP 1,23 s ; 8 requêtes simultanées en 1,54 s). L’URL de cette session n’existe plus ; chaque lancement en crée une nouvelle.**

[← 05 · Déploiement](05_deploiement.ipynb) · [Sommaire](../README.md)

Je complète la preuve d’inférence interne du notebook 06 par un appel à une URL HTTPS publique, protégée par clé. Ce service utilise réellement **FastAPI + vLLM**, avec le SFT mixte époque 3 retenu, le même adaptateur que le Space ; le Space Gradio reste une autre voie, sous Transformers/ZeroGPU.

```text
Navigateur → HTTPS temporaire (Cloudflare) → FastAPI / Bearer
                                            ↓ UUID commun
                                         vLLM → contrôle → SQLite → réponse
```

Je choisis une L4, une G4 ou une A100 (le modèle occupe moins de 4 Go de VRAM ; une L4 de 24 Go suffit), autorise Drive et passe `LANCER_ENDPOINT` à `True`. La cellule installe l’environnement, vérifie les poids et le binaire du tunnel, puis demande une clé masquée (au moins 24 caractères). Je garde cette clé pour la saisir dans la page de démo ; je ne la partage pas dans les logs, le chat ou le support public.

Le paquet `ocr14_endpoint_soutenance_20261110.zip` doit être dans le dossier Drive `OCR_PROJET_14`. Le lancement affiche `DEMO_URL=...` après trois appels fictifs vérifiés et une mesure de débit sur huit appels simultanés à travers l’URL HTTPS. J’ouvre cette URL, saisis la clé et montre un parcours. Les journaux enregistrent les appels avec leurs identifiants, sans clé ni corps dans les logs techniques.

**Durée et coût.** Le GPU reste alloué pendant installation, chargement et démonstration. La fenêtre de démo est limitée à 60 minutes après vérification, avant l’arrêt automatique du service. Interrompre la cellule déclenche l’arrêt des services, la sauvegarde et la libération du GPU. Ce n’est pas un hébergement permanent ; le tunnel ne garantit pas de disponibilité et son URL change à chaque lancement.

Je ne présente le livrable endpoint public comme vérifié qu’après lecture du rapport `verification_publique.json` et un essai depuis un navigateur extérieur. Si le tunnel ou vLLM échoue, je garde le statut d’échec et les logs. Les pages d’explication du Space restent accessibles sans cette session.

Sources : [Quick Tunnels](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/) · [vLLM](https://docs.vllm.ai/en/latest/serving/openai_compatible_server/).


### Comment lire ce carnet

Je distingue **les calculs exécutés**, **les commandes de reproduction** et **la relecture de fichiers déjà produits**. Une sortie enregistrée sous une cellule n'atteste pas une nouvelle exécution. Les chemins `data/` et `models/` sont relatifs à la racine du projet ; ces fichiers volumineux ne sont pas tous versionnés sur GitHub. Une copie du code seule ne suffit donc pas à rejouer les expériences.

La lecture suit la démarche et ses résultats ; elle ne suppose pas de lancer « Tout exécuter ». Les versions historiques sont conservées pour expliquer les corrections, et ne sont pas automatiquement celles du modèle finalement déployé.


### Ce que produit ce dernier carnet

**Entrées déjà produites :** le paquet Colab déposé sur Drive hors notebook, le socle et l'adaptateur SFT mixte époque 3 sélectionné dans 07 après rejet du DPO dans 08. Ce carnet ne réentraîne et ne sélectionne aucun modèle.

```text
paquet + poids vérifiés → installation → démarrage vLLM et FastAPI
    → tunnel HTTPS → requêtes de vérification → fenêtre de démonstration
    → sauvegarde sur Drive → arrêt du service et libération du GPU
```

`LANCER_ENDPOINT = False` garde le service arrêté. Une activation lance réellement la chaîne : la cellule affiche l'URL seulement après vérification. Les résultats sont d'abord écrits dans `/content/endpoint_soutenance_<date>/`, puis copiés dans `MyDrive/OCR_PROJET_14/evaluations/endpoint_soutenance_<date>/` avec contrôle des empreintes. Ce dossier conserve le rapport de vérification, le statut, les journaux et la base d'interactions ; ce sont les preuves à relire une fois la session terminée.

Les mesures du titre proviennent de la session du 10 novembre. Lire le carnet ne rend pas cette URL de nouveau accessible : une nouvelle exécution produit une nouvelle URL et de nouvelles traces. Le Space et ce service vLLM sont deux déploiements distincts du même adaptateur.


In [ ]:
LANCER_ENDPOINT = False  # True uniquement pour une démonstration GPU.
DUREE_DEMO_MINUTES = 60  # Arrêt automatique après vérification ; 1 à 60 minutes. 60 couvre les 30 minutes d'oral.
if LANCER_ENDPOINT:
    from google.colab import drive, runtime
    from getpass import getpass
    from pathlib import Path
    from datetime import datetime, timezone
    import hashlib, os, shutil, signal, subprocess, sys, urllib.request, zipfile
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/OCR_PROJET_14')
    RACINE = Path('/content/OCR14_endpoint_soutenance')
    RACINE.mkdir(parents=True, exist_ok=True)
    identifiant = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    SORTIES = Path('/content') / f'endpoint_soutenance_{identifiant}'
    processus = None
    try:
        print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True).strip())
        paquet = DRIVE / 'ocr14_endpoint_soutenance_20261110.zip'
        assert hashlib.sha256(paquet.read_bytes()).hexdigest() == '1d830c5097b80ffef43692a51ccb20547cfae8421429042878a04b24088f7988', 'Paquet différent'
        with zipfile.ZipFile(paquet) as z:
            assert all((RACINE / n).resolve().is_relative_to(RACINE.resolve()) for n in z.namelist())
            z.extractall(RACINE)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
        ENV = '/content/ocr14_endpoint_env'
        subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '--seed', '--managed-python', ENV], check=True)
        PYTHON = ENV + '/bin/python'
        subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
                        'vllm==0.32.0', 'uvicorn==0.54.0', 'jinja2==3.1.6', '--torch-backend=auto'], check=True)
        subprocess.run([ENV + '/bin/vllm', 'download-kernels'], check=True)
        # Poids publics déjà déployés, révision figée ; le lanceur vérifie aussi leur SHA.
        telecharger = "from huggingface_hub import hf_hub_download; " + "; ".join(
            f"hf_hub_download('rayakevin/ocr14-triage-fr-en', 'adapter/{nom}', repo_type='space', revision='f32b2a65f9ef05fb431b4e07d65253d82b5d049a', local_dir='{RACINE / 'poids'}')"
            for nom in ('adapter_config.json', 'adapter_model.safetensors'))
        subprocess.run([PYTHON, '-c', telecharger], check=True)
        tunnel = RACINE / 'cloudflared'
        urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64', tunnel)
        assert hashlib.sha256(tunnel.read_bytes()).hexdigest() == '1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872'
        tunnel.chmod(0o700)
        cle = getpass('Clé temporaire (24 caractères minimum ; à saisir ensuite dans la page de démo) : ')
        if len(cle) < 24:
            raise ValueError('Clé trop courte ; aucune API exposée.')
        commande = [PYTHON, '-m', 'serving.servir_demo', '--adapter', str(RACINE / 'poids/adapter'),
                    '--cloudflared', str(tunnel), '--output', str(SORTIES), '--minutes', str(DUREE_DEMO_MINUTES)]
        processus = subprocess.Popen(commande, cwd=RACINE, env={**os.environ, 'OCR14_API_KEY': cle, 'PYTHONUNBUFFERED': '1'},
                                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)
        del cle
        for ligne in processus.stdout:
            print(ligne, end='', flush=True)
        if processus.wait():
            raise RuntimeError('Échec de la démonstration ; voir les journaux sauvegardés.')
    finally:
        if processus is not None and processus.poll() is None:
            # SIGINT laisse le lanceur fermer vLLM, l’API et le tunnel dans son finally.
            os.killpg(processus.pid, signal.SIGINT)
            try: processus.wait(timeout=90)
            except subprocess.TimeoutExpired:
                os.killpg(processus.pid, signal.SIGKILL)
                processus.wait()
        if SORTIES.exists():
            destination = DRIVE / 'evaluations' / SORTIES.name
            shutil.copytree(SORTIES, destination, dirs_exist_ok=True)
            for f in SORTIES.rglob('*'):
                if f.is_file():
                    assert hashlib.sha256(f.read_bytes()).digest() == hashlib.sha256((destination / f.relative_to(SORTIES)).read_bytes()).digest(), 'Sauvegarde incomplète : je conserve la VM'
            print('Preuves sauvegardées dans', destination)
        print('Synchronisation Drive et libération du GPU.')
        drive.flush_and_unmount(timeout_ms=300_000)
        runtime.unassign()
else:
    print('Endpoint désactivé. Aucun GPU ni tunnel démarré.')
